# Experiment Tracking - Practical Examples
## Learn how to track ML experiments with MLflow

## Part 1: Why Experiment Tracking Matters

In [ ]:
import pandas as pd
import numpy as np

print("=== EXPERIMENT TRACKING IMPORTANCE ===")
print()

print("WITHOUT Experiment Tracking:")
print("-" * 50)

experiments_without = [
    'model.pkl',
    'model_v2.pkl',
    'model_final.pkl',
    'model_final_v2.pkl',
    'model_FINAL_REAL.pkl',
    'model_best.pkl',
    'model_old.pkl',
    'model_backup.pkl',
    'model_production.pkl',
]

for i, model_name in enumerate(experiments_without, 1):
    print(f"{i}. {model_name}")

print()
print("Problems:")
print("❌ Which model is best?")
print("❌ What parameters were used?")
print("❌ When was it trained?")
print("❌ What accuracy does it have?")
print("❌ How does it compare?")
print("❌ No way to reproduce results")
print()
print()

print("WITH Experiment Tracking (MLflow):")
print("-" * 50)

experiments_with = pd.DataFrame({
    'Experiment': ['exp_001', 'exp_002', 'exp_003', 'exp_004', 'exp_005'],
    'Algorithm': ['Logistic Regression', 'SVM', 'Random Forest', 'Gradient Boosting', 'XGBoost'],
    'Accuracy': [0.92, 0.94, 0.96, 0.97, 0.96],
    'Precision': [0.91, 0.93, 0.95, 0.96, 0.95],
    'Timestamp': ['2024-05-01 10:00', '2024-05-01 11:00', '2024-05-01 12:00', '2024-05-01 13:00', '2024-05-01 14:00']
})

print(experiments_with.to_string(index=False))
print()
print("Best: exp_004 - Gradient Boosting (97% accuracy)")
print()
print("Benefits:")
print("✓ Clear comparison")
print("✓ Know which is best")
print("✓ Reproducible")
print("✓ Team collaboration")
print("✓ Historical records")

## Part 2: Setting up MLflow

In [ ]:
print("\n=== MLFLOW SETUP ===")
print()

print("Installation:")
print("pip install mlflow")
print()

print("Basic MLflow example:")
print()

mlflow_code = '''
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Create data
X, y = make_classification(n_samples=1000)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Start MLflow run
with mlflow.start_run():
    # Log parameters
    mlflow.log_param("algorithm", "Random Forest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 15)
    
    # Train model
    model = RandomForestClassifier(n_estimators=100, max_depth=15)
    model.fit(X_train, y_train)
    
    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    
    # Log model
    mlflow.sklearn.log_model(model, "model")
    
    print(f"Accuracy: {accuracy}")
    print("Experiment logged!")

# View experiments
mlflow ui  # Opens http://localhost:5000
'''

print(mlflow_code)

## Part 3: Tracking Multiple Experiments

In [ ]:
print("\n=== TRACKING MULTIPLE EXPERIMENTS ===")
print()

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Create data
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define experiments to run
experiments = [
    {
        'name': 'Logistic Regression',
        'model': LogisticRegression(),
        'params': {'algorithm': 'Logistic Regression', 'max_iter': 1000}
    },
    {
        'name': 'SVM',
        'model': SVC(probability=True),
        'params': {'algorithm': 'SVM', 'kernel': 'rbf'}
    },
    {
        'name': 'Random Forest',
        'model': RandomForestClassifier(n_estimators=100, random_state=42),
        'params': {'algorithm': 'Random Forest', 'n_estimators': 100, 'max_depth': 15}
    },
    {
        'name': 'Gradient Boosting',
        'model': GradientBoostingClassifier(n_estimators=100, random_state=42),
        'params': {'algorithm': 'Gradient Boosting', 'n_estimators': 100, 'learning_rate': 0.1}
    },
]

print("Training and tracking experiments:")
print("-" * 70)

results = []

for exp in experiments:
    print(f"\nTraining {exp['name']}...")
    
    # Train model
    model = exp['model']
    model.fit(X_train, y_train)
    
    # Make predictions
    predictions = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    
    # Store results
    results.append({
        'Algorithm': exp['name'],
        'Accuracy': f'{accuracy:.2%}',
        'Precision': f'{precision:.2%}',
        'Recall': f'{recall:.2%}',
        'Raw Accuracy': accuracy
    })
    
    print(f"  ✓ Accuracy: {accuracy:.2%}")
    print(f"  ✓ Precision: {precision:.2%}")
    print(f"  ✓ Recall: {recall:.2%}")

print()
print("\nExperiment Results:")
print("-" * 70)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

best = results_df.loc[results_df['Raw Accuracy'].idxmax()]
print(f"\nBest Model: {best['Algorithm']} (Accuracy: {best['Accuracy']})")

## Part 4: Hyperparameter Tuning with Tracking

In [ ]:
print("\n=== HYPERPARAMETER TUNING WITH TRACKING ===")
print()

print("Testing different hyperparameters:")
print("-" * 70)

hp_results = []

for n_trees in [50, 100, 150, 200]:
    for depth in [10, 15, 20, 25]:
        # Train with different hyperparameters
        model = RandomForestClassifier(
            n_estimators=n_trees,
            max_depth=depth,
            random_state=42
        )
        model.fit(X_train, y_train)
        
        accuracy = model.score(X_test, y_test)
        
        hp_results.append({
            'n_trees': n_trees,
            'max_depth': depth,
            'accuracy': accuracy
        })

hp_df = pd.DataFrame(hp_results)
best_hp = hp_df.loc[hp_df['accuracy'].idxmax()]

print("Top 5 hyperparameter combinations:")
print(hp_df.nlargest(5, 'accuracy').to_string(index=False))

print()
print(f"Best hyperparameters:")
print(f"  n_estimators: {int(best_hp['n_trees'])}")
print(f"  max_depth: {int(best_hp['max_depth'])}")
print(f"  Accuracy: {best_hp['accuracy']:.2%}")

## Part 5: Logging Artifacts

In [ ]:
print("\n=== LOGGING ARTIFACTS ===")
print()

print("Beyond metrics, MLflow can log:")
print()

artifacts = {
    'Models': 'Trained model files (.pkl, .h5)',
    'Plots': 'Visualizations (confusion matrix, ROC curve)',
    'Data': 'Training/test data splits',
    'Reports': 'Text reports and summaries',
    'Metadata': 'JSON files with additional info',
    'Code': 'Scripts used for training'
}

for artifact_type, description in artifacts.items():
    print(f"✓ {artifact_type:15} - {description}")

print()
print("Example MLflow code for logging artifacts:")
print()

artifact_code = '''
import mlflow
import matplotlib.pyplot as plt
import pickle

with mlflow.start_run():
    # ... train model ...
    
    # Log model
    mlflow.sklearn.log_model(model, "model")
    
    # Log plots
    plt.figure()
    plt.plot([1, 2, 3, 4], [1, 4, 2, 3])
    plt.savefig('plot.png')
    mlflow.log_artifact('plot.png')
    
    # Log custom data
    with open('report.txt', 'w') as f:
        f.write(f'Model trained with {len(X_train)} samples')
    mlflow.log_artifact('report.txt')
'''

print(artifact_code)

## Summary

1. **Experiment Tracking** - Essential for reproducibility
2. **MLflow** - Free, easy-to-use tool
3. **Log Everything** - Parameters, metrics, artifacts
4. **Compare Experiments** - Find best model systematically
5. **Team Collaboration** - Shared experiment history

Next: Learn about CI/CD in Section 5!